# 🤝 Building specialist teams

- We'll be building a series of specialized agents, one for each role we're planning to have in our team.

- Think of these agents as factories for each content type:
    - We're using functions to create specialized agent pairs
    - Each content type gets its own researcher + writer team
    - This allows for highly specialized expertise and output formats

<img src="images/agents_roles.png" width="600" alt="Agents each role: blog, newsletter, LinkedIn">

### ❗️ Note: Run the **hidden cells** below before running the rest of the code. ❗️ 

In [15]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-U",
                "crewai==0.157.0", "crewai_tools==0.58.0"],
               check=True)

  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_community-0.3.31-py3-none-any.whl (2.5 MB)
Using cached langchain_core-0.3.83-py3-none-any.whl (458 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.14
    Uninstalling langchain-core-1.2.14:
      Successfully uninstalled langchain-core-1.2.14
  Attempting uninstall: langchain-text-splitters 0/3 [langchain-core]
    Found existing installation: langchain-text-splitters 1.1.1langchain-core]
    Uninstalling langchain-text-splitters-1.1.1: 0/3 [langchain-core]
      Successfully uninstalled langchain-text-splitters-1.1.1 [langchain-core]
  Attempting uninstall: langchain-community━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [langchain-text-splitters]
    Found existing installation: langchain-community 0.4.1━━━━ 1/3 [langchain-text-splitters]
    Uninstalling langchain-community-0.4.1:━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.1 requires langchain-core<2.0.0,>=1.2.5, but you have langchain-core 0.3.83 which is incompatible.
langchain-classic 1.0.1 requires langchain-text-splitters<2.0.0,>=1.1.0, but you have langchain-text-splitters 0.3.11 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/home/repl/.venv-user/bin/python', '-m', 'pip', 'install', '-U', 'crewai==0.157.0', 'crewai_tools==0.58.0'], returncode=0)

In [16]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-U",
                "ddgs", "duckduckgo-search", "langchain-community"],
               check=True)

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.14-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_core-1.2.14-py3-none-any.whl (501 kB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
  Attempting uninstall: langchain-text-splitters 0/3 [langchain-core]
    Found existing installation: langchain-text-splitters 0.3.11angchain-core]
    Uninstalling langchain-text-splitters-0.3.11:0/3 [langchain-core]
      Successfully uninstalled langchain-text-splitters-0.3.11[langchain-core]
  Attempting uninstall: langchain-community━ 0/3 [langchain-core]
    Found existing instal

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.2.14 requires langchain-core<0.4.0,>=0.3.27, but you have langchain-core 1.2.14 which is incompatible.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.2.14 which is incompatible.
langchain-cohere 0.3.5 requires langchain-core<0.4.0,>=0.3.27, but you have langchain-core 1.2.14 which is incompatible.
embedchain 0.1.128 requires langchain-community<0.4.0,>=0.3.1, but you have langchain-community 0.4.1 which is incompatible.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.2.14 which is incompatible.
langchain 0.3.27 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have l

CompletedProcess(args=['/home/repl/.venv-user/bin/python', '-m', 'pip', 'install', '-U', 'ddgs', 'duckduckgo-search', 'langchain-community'], returncode=0)

In [17]:
from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import tool

# Initialize the search tool
search_tool_instance = DuckDuckGoSearchRun()

# Wrap the langchain tool with the CrewAI @tool decorator
@tool("DuckDuckGo Search")
def search_tool(query: str) -> str:
    """A tool for searching the web using DuckDuckGo."""
    return search_tool_instance.run(query)

### 🤖🤖 Define specialized agent pairs

**1) Blog content creation agents**

In [18]:
from crewai import Agent, LLM

AGENT_LLM = LLM(model="openai/gpt-4o-mini")

def create_blog_agents():
    """Create agents specialized for blog content"""

    blog_researcher = Agent(
        role="Blog Content Researcher",
        goal="Extract and analyze web content to identify key insights for blog posts",
        backstory="""You are an expert content researcher who specializes in analyzing
        web content and identifying the most valuable insights for creating engaging blog posts.
        You excel at understanding complex topics and breaking them down into digestible content.""",
        verbose=False,
        tool=[search_tool], # CODE: Add a search tool to the researcher agent
        llm=AGENT_LLM,
        max_iter=5
    )

    blog_writer = Agent(
        role="Blog Content Writer",
        goal="Transform research into engaging, well-structured blog posts",
        backstory="""You are a skilled blog writer with expertise in creating compelling content
        that engages readers and drives meaningful discussions. You excel at taking complex
        information and making it accessible and interesting.""",
        verbose=False,
        llm=AGENT_LLM
    )

    return blog_researcher, blog_writer

**2) Newsletter content creation agents**

In [19]:
def create_newsletter_agents():
    """Create agents specialized for newsletter content"""

    newsletter_researcher = Agent(
        role="Newsletter Content Researcher",
        goal="Extract key insights from web content for newsletter format",
        backstory="""You are an expert at identifying the most newsworthy and actionable
        insights from web content. You understand what makes content valuable for newsletter
        subscribers and how to present information concisely.""",
        verbose=False,
        tool=[search_tool],
        llm=AGENT_LLM,
        max_iter=5
    )

    newsletter_writer = Agent(
        role="Newsletter Writer",
        goal="Create engaging newsletter content that provides immediate value",
        backstory="""You are a newsletter specialist who knows how to craft content that
        busy professionals want to read. You excel at creating scannable, actionable content
        with clear takeaways.""",
        verbose=False,
        llm=AGENT_LLM
    )

    return newsletter_researcher, newsletter_writer

**3) LinkedIn content creation agents**

In [20]:
def create_linkedin_agents():
    """Create agents specialized for LinkedIn content"""

    linkedin_researcher = Agent(
        role="LinkedIn Content Researcher",
        goal="Extract professional insights suitable for LinkedIn audience",
        backstory="""You are an expert at identifying professional insights and industry
        trends that resonate with LinkedIn's professional audience. You understand what
        content drives engagement on professional networks.""",
        verbose=False,
        tool=[search_tool],
        llm=AGENT_LLM,
        max_iter=5
    )

    linkedin_writer = Agent(
        role="LinkedIn Content Writer",
        goal="Create engaging LinkedIn posts that drive professional engagement",
        backstory="""You are a LinkedIn content specialist who knows how to craft posts
        that get noticed in the professional feed. You excel at creating content that
        sparks meaningful professional discussions.""",
        verbose=False,
        llm=AGENT_LLM
    )

    return linkedin_researcher, linkedin_writer

### 📝 Define task creation functions for each content type

Task design principles:
1. Clear, specific instructions improve output quality
2. One-shot examples of the final output to help the agent

**1) Tasks for blog writing**

In [21]:
def create_blog_tasks(researcher, writer, url):
    """Create tasks for blog content generation"""

    # CODE: Define the blog research task
    research_task = Task(
        description=f"""
        Analyze the content from {url} and extract key insights for a blog post.
        Your analysis should identify:
        1. Main themes and key points
        2. Interesting insights or data points
        3. Potential angles for blog content
        4. Target audience considerations
        5. SEO-worthy topics and keywords

        Provide a comprehensive research summary that will guide blog writing.
        """,
        expected_output="A detailed research summary with key insights, themes, and recommendations for blog content",
        agent=researcher 
    )

    writing_task = Task(
        description="""
        Create an engaging blog post based on the research findings.

        Requirements:
        - 800-1200 words
        - Engaging headline
        - Clear introduction with hook
        - Well-structured body with subheadings
        - Actionable insights or takeaways
        - Strong conclusion
        - SEO-optimized content
        - Professional yet accessible tone

        Format the output in markdown.
        """,
        expected_output="A complete, well-structured blog post in markdown format",
        agent=writer,
        # CODE: add the research task as context to the writing task
        context=[research_task]
    )

    return [research_task, writing_task]

**2) Tasks for newsletter content creation**

In [22]:
def create_newsletter_tasks(researcher, writer, url):
    """Create tasks for newsletter content generation"""

    research_task = Task(
        description=f"""
        Analyze the content from {url} and extract the most newsworthy insights for a newsletter.
        Focus on:
        1. Most important news or updates
        2. Actionable insights subscribers can use immediately
        3. Key statistics or data points
        4. Industry implications
        5. Quick takeaways for busy professionals

        Prioritize information that provides immediate value.
        """,
        expected_output="A focused research summary highlighting the most valuable and actionable insights",
        agent=researcher
    )

    writing_task = Task(
        description="""
        Create a compelling newsletter section based on the research.

        Requirements:
        - 400-600 words
        - Catchy subject line
        - Scannable format with bullet points
        - Clear action items or takeaways
        - Conversational yet professional tone
        - Include relevant links or resources
        - End with a clear call-to-action

        Format for easy reading in email.
        """,
        expected_output="A complete newsletter section with subject line and formatted content",
        agent=writer,
        context=[research_task]
    )

    return [research_task, writing_task]

**3) Tasks for LinkedIn post content creation**

In [23]:
def create_linkedin_tasks(researcher, writer, url):
    """Create tasks for LinkedIn content generation"""

    research_task = Task(
        description=f"""
        Analyze the content from {url} and extract insights suitable for LinkedIn audience.
        Consider what would engage LinkedIn's professional audience based on the content.
        """,
        expected_output="Research summary focused on professional insights and engagement opportunities",
        agent=researcher
    )

    writing_task = Task(
        description="""
        Create an engaging LinkedIn post based on the research.

        Requirements:
        - 150-300 words (optimal LinkedIn length)
        - Professional yet conversational tone
        - Include relevant hashtags (3-5)
        - Pose a question to encourage engagement
        - Share a key insight or lesson learned from the content
        - Use line breaks for readability
        - Include a call-to-action for comments

        Make it shareable and discussion-worthy.
        """,
        expected_output="A complete LinkedIn post with hashtags and engagement elements",
        agent=writer,
        context=[research_task]
    )

    return [research_task, writing_task]